# 三维道路 DAS + 锤击地下空洞探测算法原型

本 Notebook 是项目进度把控入口，教学说明是第二作用。当前阶段是 **Stage 2A：从三维运动学绕射原型过渡到真实三维波动方程正演后端准备阶段**。

当前不要把定位误差调参作为核心目标。定位模块目前用于验证三维几何、观测系统、正演数据结构和定位流程是否贯通。

## 1. 当前项目状态摘要

- 当前默认正演后端：`kinematic`。
- Devito 当前在 `myvoid` 中不可 import。
- OpenSWPC 当前未编译、未配置可执行文件。
- 已新增统一 `ForwardBackend` 接口。
- 已新增 `VoidBody3D` 体异常模型和 `VelocityGrid3D` 三维速度网格。
- 当前仍不是完整三维声波/弹性波正演。
- 当前 DAS 仍是 polyline 点采样近似，不是真实 gauge-length averaged axial strain。

## 2. 当前默认三维道路场景

坐标系统：

- `x`：沿道路或 DAS 光纤方向；
- `y`：横穿道路方向；
- `depth`：向下为正。

默认几何：

- 道路长度：80 m；
- 道路宽度：15 m；
- DAS 光纤：`receiver_polyline=[(0,0,0),(80,0,0)]`；
- 锤击炮线：`y=15 m`，沿 `x=0~80 m` 排列；
- 空洞中心：`void_xyz=(40.0, 7.5, 2.0)`；
- 体异常代理：椭球体 `size_xyz_m=(2.0, 2.0, 1.0)`，`velocity_scale=0.5`。

光纤、锤击点和空洞不在同一条线上，因此这是三维观测几何，不是二维剖面。

In [ ]:
from pathlib import Path
import json

summary_path = Path('../outputs/run_summary.json')
if summary_path.exists():
    summary = json.loads(summary_path.read_text(encoding='utf-8'))
    print('backend_name =', summary['backend_name'])
    print('true_xyz =', summary['true_xyz'])
    print('best_xyz =', summary['best_xyz'])
    print('localization_error_m =', summary['localization_error_m'])
    print('velocity_grid_shape =', summary['velocity_grid']['shape'])
else:
    print('尚未找到 ../outputs/run_summary.json，请先在 code/ 目录运行 python main.py。')

## 3. 当前正演短板

当前运动学正演只计算总走时：

`总走时 = 震源到空洞的传播时间 + 空洞到接收点的传播时间`

即：

`t_total = |source_xyz - void_xyz| / v + |void_xyz - receiver_xyz| / v`

这个模型可以检查三维几何和定位流程，但不能模拟自由表面、PML、反射、转换波、体散射、波场干涉、衰减、各向异性或真实 DAS 应变。项目不能长期停留在这个层级。

## 4. 本地 tools 二次审计结论摘要

- Devito：优先候选，Python API 适合先接入三维声波方程和快照。
- OpenSWPC：高保真外部程序候选，适合后续弹性/黏弹性和 DAS 应变。
- SW4：地表弹性波能力强，可作为外部对照。
- SPECFEM3D：物理能力强，但网格和编译成本高。
- SAVA、SeisCL：能力强但 GPL 和编译链较重。
- SOFI2D、SPECFEM1D：不是三维主后端。

详细记录见 `code/docs/TOOLS_DEEP_DIVE.md`。

## 5. Devito 接入路线

当前 `myvoid` 中 Devito 不可用。下一轮建议：

1. 在 `myvoid` 中安装并验证 Devito；
2. 将 `VelocityGrid3D` 转为 Devito `Grid` 和 `Model`；
3. 将 `source_xyz` 转为 `RickerSource` 或 `SparseTimeFunction`；
4. 将 `receiver_xyz` 转为 `Receiver`；
5. 输出 `ForwardResult3D`、真实声波场快照和动图。

第一版 Devito 后端只做 3D acoustic，不做真实 DAS 轴向应变。

## 6. OpenSWPC 接入路线

OpenSWPC 更适合作为外部高保真正演程序：

`本项目生成速度模型、震源文件、接收点文件、参数文件 -> 调用 OpenSWPC 外部程序 -> 读取 SAC/NetCDF -> 转换为 ForwardResult3D`

它对后续真实 DAS 轴向应变更有价值，因为可以输出位移、速度、应力或应变相关数据。当前未编译，未配置可执行文件。

## 7. 异常体表示推进路线

当前有三层表示：

1. `VoidModel3D`：中心点和半径，用于当前运动学绕射；
2. `sample_void_body_as_scatterers()`：把体异常采样成多散射点代理；
3. `VelocityGrid3D.with_embedded_void_body()`：把体异常写入三维速度网格。

多散射点代理不是严格边界散射。后续真实波动方程后端应直接使用体速度模型。

## 8. 三维速度模型输出路线

当前 `main.py` 输出：

- `outputs/velocity_model_3d.npz`
- `outputs/velocity_model_slices.png`

速度网格轴顺序为 `x-y-depth`，`vp_mps` 形状当前为 `[81, 16, 9]`。后续弹性波后端需要扩展 `vs`、`rho`、Q 和可能的各向异性参数。

## 9. 炮集、波场快照和动图输出路线

当前已有炮集输出和目录约定：

- `synthetic_gather.png`：当前运动学炮集；
- `wavefield_snapshots/`：后续真实波场快照目录；
- `wavefield_animation.gif`：后续真实波场动图目标。

当前 `wavefield_snapshot_type='not_available'`，`is_true_wave_equation_wavefield=False`。不要把运动学示意图冒充真实波场。

## 10. DAS 光纤观测的当前近似

`ReceiverPolyline3D` 已能记录 gauge length 和局部切向量，但当前合成记录仍是点接收器近似。真正 DAS 轴向应变需要弹性位移场或应变张量，并沿光纤切向量计算：

`epsilon_fiber = t_i * epsilon_ij * t_j`

这要等弹性波后端稳定后再做。

## 11. 为什么当前不追求定位精度

当前定位误差不能代表工程精度，因为正演仍是运动学点绕射近似。若现在为了让 `best_xyz` 更接近 `true_xyz` 而硬调目标函数，会把误差降低建立在不真实的波形模型上。

当前验收重点是：三维几何、速度模型、正演后端接口、炮集数据结构、波场输出约定和定位流程是否贯通。

## 12. 当前最新 main.py 输出

最近一次 `myvoid` 运行结果：

- 当前正演后端：`kinematic`；
- Devito 可用：`False`；
- OpenSWPC 可用：`False`；
- `true_xyz=(40.0, 7.5, 2.0)`；
- `best_xyz=(40.0, 7.5, 1.5)`；
- `localization_error_m=0.5`；
- 数据维度：`[21, 41, 600]`；
- 目标函数维度：`[31, 31, 19]`；
- 速度网格维度：`[81, 16, 9]`。

这些结果只说明当前闭环可运行，不代表真实工程定位精度。

## 13. 当前限制

1. 默认正演仍是三维运动学点绕射近似。
2. 尚未实现 Devito/OpenSWPC 真实正演。
3. 当前没有真实波场快照和动图。
4. 当前 DAS 不是真实 gauge-length averaged axial strain。
5. 体异常已能进入速度网格，但尚未由波动方程求解器使用。
6. 单侧光纤和单侧锤击几何仍可能造成 `y-depth` 混淆。

## 14. 下一轮计划

1. 在 `myvoid` 中安装并验证 Devito。
2. 实现最小 3D acoustic Devito 后端。
3. 使用 `VelocityGrid3D` 作为 Devito 模型输入。
4. 输出真实声波炮集、波场快照和动图。
5. 更新 Notebook，清理运动学阶段过期结论。
6. 在真实正演稳定后再系统优化定位目标函数。